# Plate Number Text Extraction

In [ ]:
import os
import re
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
from pytesseract import Output

# --- find the Tesseract program ---
exe = shutil.which("tesseract")
if exe is None:
    for p in (r"C:\Program Files\Tesseract-OCR\tesseract.exe",
              r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"):
        if os.path.exists(p):
            exe = p
            break
if exe is None:
    raise RuntimeError("Tesseract is not installed or not on PATH. Install it, or set exe to its full path.")
pytesseract.pytesseract.tesseract_cmd = exe
print("Using Tesseract:", exe)

# --- the images to read ---
PLATES = [
    ("Private",        "Images/pn-private.png"),
    ("Public Utility", "Images/pn-public_utility.png"),
    ("Government",     "Images/pn-government.png"),
    ("Diplomatic",     "Images/pn-diplomatic.png"),
    ("Others",         "Images/pn-others.png"),
]

CHARS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
OCR_CONFIG = f"--oem 3 --psm 7 -c tessedit_char_whitelist={CHARS}"   # psm 7 = one line of text
TARGET_W = 1200      # every plate is resized to this width first
TEXT_H = 90          # the number line is scaled to this height (pixels) for OCR

In [ ]:
def load_plate(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {path}")
    s = TARGET_W / img.shape[1]
    interp = cv2.INTER_CUBIC if s > 1 else cv2.INTER_AREA
    return cv2.resize(img, None, fx=s, fy=s, interpolation=interp)

def to_ink(img):
    """Return the smoothed gray image and a binary mask where text pixels are 255."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    smooth = cv2.GaussianBlur(gray, (0, 0), 1.5)
    smooth = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(smooth)
    _, bw = cv2.threshold(smooth, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    ink = bw if cv2.countNonZero(bw) < bw.size / 2 else cv2.bitwise_not(bw)
    return smooth, ink

In [ ]:
def find_number_line(ink):
    H, W = ink.shape
    n, lab, stats, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)

    allc, cand = [], []
    for i in range(1, n):
        x, y, w, h, a = stats[i]
        if x <= 1 or y <= 1 or x + w >= W - 1 or y + h >= H - 1:   # touches the edge: plate frame
            continue
        if w > 0.5 * W or h > 0.6 * H:                              # too big to be a character
            continue
        allc.append((i, x, y, w, h))
        if a >= 0.0003 * H * W and h >= 0.03 * H and w >= 0.1 * h:  # not a speck, not a thin bar
            cand.append((i, x, y, w, h))

    # group candidates by row; the row with the tallest characters wins
    best = None
    for (_, x, y, w, h) in cand:
        yc = y + h / 2
        row = [c for c in cand if abs(c[2] + c[4] / 2 - yc) < 0.3 * h and 0.7 * h < c[4] < 1.4 * h]
        if len(row) < 3:
            continue
        med_h = float(np.median([c[4] for c in row]))
        if best is None or med_h > best[0]:
            best = (med_h, row)
    if best is None:
        return None

    line_h, row = best
    yc_line = float(np.median([c[2] + c[4] / 2 for c in row]))
    gx0 = min(c[1] for c in row)
    gx1 = max(c[1] + c[3] for c in row)

    # keep everything on that row, including dashes and dots, but not screws far outside the characters
    members = [c for c in allc
               if abs(c[2] + c[4] / 2 - yc_line) < 0.6 * line_h
               and c[4] <= 1.5 * line_h
               and gx0 - 0.4 * line_h <= c[1] + c[3] / 2 <= gx1 + 0.4 * line_h]

    x0 = min(c[1] for c in members);        x1 = max(c[1] + c[3] for c in members)
    y0 = min(c[2] for c in members);        y1 = max(c[2] + c[4] for c in members)
    mask = np.isin(lab, [c[0] for c in members])
    return {"box": (x0, y0, x1, y1), "mask": mask}

In [ ]:
def ocr_images(smooth, ink, found):
    x0, y0, x1, y1 = found["box"]
    pad = int(0.35 * (y1 - y0))
    s = TEXT_H / (y1 - y0)

    # version A: only the number-line pixels, black text on white
    binary = (found["mask"] * 255).astype(np.uint8)[y0:y1, x0:x1]
    # version B: the gray picture of the same area, with dark text on a light background
    gray = smooth[y0:y1, x0:x1].copy()
    ink_here = found["mask"][y0:y1, x0:x1]
    if gray[ink_here].mean() > gray[~ink_here].mean():
        gray = 255 - gray
    gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)

    out = []
    for im, white_bg in ((binary, False), (gray, True)):
        if not white_bg:
            im = 255 - im                                  # black text on white
        im = cv2.copyMakeBorder(im, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=255)
        im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
        out.append(cv2.GaussianBlur(im, (0, 0), 0.8))
    return out

In [ ]:
def run_ocr(im):
    d = pytesseract.image_to_data(im, config=OCR_CONFIG, output_type=Output.DICT)
    words, confs = [], []
    for t, c in zip(d["text"], d["conf"]):
        if t.strip() and float(c) >= 0:
            words.append(t.strip())
            confs.append(float(c))
    return " ".join(words), (float(np.mean(confs)) if confs else 0.0)

TO_LETTER = {"0": "O", "1": "I", "2": "Z", "5": "S", "6": "G", "8": "B"}
TO_DIGIT  = {"O": "0", "D": "0", "Q": "0", "I": "1", "L": "1", "Z": "2", "S": "5", "G": "6", "B": "8"}

def fix_characters(text):
    """Plates group letters and digits separately, so each group is made consistent."""
    text = re.sub(r"[^A-Z0-9 ]", "", text.upper())
    fixed = []
    for tok in text.split():
        letters = sum(ch.isalpha() for ch in tok)
        digits = len(tok) - letters
        table = TO_LETTER if letters >= digits else TO_DIGIT
        fixed.append("".join(table.get(ch, ch) for ch in tok))
    return " ".join(fixed)

In [ ]:
def read_plate(label, path, show=True):
    img = load_plate(path)
    smooth, ink = to_ink(img)
    found = find_number_line(ink)
    if found is None:
        print(f"{label}: no number line found")
        return ""

    candidates = [run_ocr(im) for im in ocr_images(smooth, ink, found)]
    raw, conf = max(candidates, key=lambda t: t[1])
    text = fix_characters(raw)

    if show:
        x0, y0, x1, y1 = found["box"]
        boxed = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        cv2.rectangle(boxed, (x0, y0), (x1, y1), (255, 0, 0), 4)
        profile = found["mask"].sum(axis=1)                       # ink per row (horizontal projection)

        fig, axs = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={"width_ratios": [2, 2, 1]})
        axs[0].imshow(boxed); axs[0].set_title(f"{label}: detected number line"); axs[0].axis("off")
        axs[1].imshow(ocr_images(smooth, ink, found)[0], cmap="gray"); axs[1].set_title("Image given to OCR"); axs[1].axis("off")
        axs[2].plot(profile, np.arange(len(profile))); axs[2].invert_yaxis()
        axs[2].set_title("Ink per row (number line only)"); axs[2].set_xlabel("pixels")
        plt.tight_layout(); plt.show()

    print(f"{label}: Extracted Text: {text}   (OCR confidence {conf:.0f}%)")
    return text

results = [(label, read_plate(label, path)) for label, path in PLATES]

print("\n==== SUMMARY ====")
for label, text in results:
    print(f"{label:<15}: {text}")